# VLA 연구 · MuJoCo 실습
## 로봇 시뮬레이션 기초 & Panda 정리 환경 만들기

| 항목 | 내용 |
|------|------|
| 오늘 목표 | MuJoCo 설치, MJCF/XML 씬 구성, offscreen 렌더링, Panda 로봇 팔/그리퍼 기초 제어 |
| 프로젝트 연결 | VLM/VLA가 사용할 로봇 실험 환경과 씬 이미지·메타데이터 생성 |
| 최종 확인 | 랜덤 정리 씬 5개를 렌더링하고 JSON 샘플 저장 |

이번 실습에서는 MuJoCo를 처음 접하는 것을 기준으로 시작합니다. 목표는 **우리 연구에서 이미지와 로봇 제어 데이터가 어떻게 만들어지는지** 직접 확인하는 것입니다.

---
## SECTION 0  |  환경 확인

### 0-1. GPU 및 Python 환경 확인
아래 셀을 실행하여 Colab 런타임 정보를 확인합니다. MuJoCo 렌더링은 CPU에서도 일부 가능하지만, 오늘 실습은 **GPU 런타임**을 권장합니다.

In [ ]:
# 한글 폰트 설정 (Colab 환경)
# 아래 블록 실행 후 런타임 재시작 필요
import matplotlib
import matplotlib.font_manager as fm

## Colab에 나눔폰트 설치
import subprocess
subprocess.run(['apt-get', 'install', '-y', 'fonts-nanum'], capture_output=True)
fm._load_fontmanager(try_read_cache=False)  # 폰트 캐시 갱신

## 나눔고딕 적용
matplotlib.rcParams['font.family'] = 'NanumGothic'
matplotlib.rcParams['axes.unicode_minus'] = False  # 마이너스 기호 깨짐 방지

In [ ]:
import sys
import os
import platform
import subprocess

print('=== 환경 정보 ===')
print(f'Python   : {sys.version.split()[0]}')
print(f'Platform : {platform.platform()}')
print(f'CWD      : {os.getcwd()}')

try:
    gpu_result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True, check=False)
    if gpu_result.returncode == 0 and gpu_result.stdout.strip():
        print(f'GPU      : {gpu_result.stdout.strip()}')
    else:
        print('GPU      : 감지되지 않음 - 런타임 유형을 확인하세요.')
except FileNotFoundError:
    print('GPU      : nvidia-smi 없음 - Colab GPU 런타임인지 확인하세요.')

### 0-2. 작업 폴더 만들기
Colab 가상 머신 안에 오늘 사용할 작업 폴더를 만듭니다. 런타임이 초기화되면 이 폴더는 사라지므로, 중요한 결과는 뒤에서 공유 폴더에 저장합니다.

In [ ]:
from pathlib import Path

WORK_DIR = Path('/content/vla_mujoco')
XML_DIR = WORK_DIR / 'xml'
IMAGE_DIR = WORK_DIR / 'images'
JSON_DIR = WORK_DIR / 'json'
GIF_DIR = WORK_DIR / 'gifs'

for path in [WORK_DIR, XML_DIR, IMAGE_DIR, JSON_DIR, GIF_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print(f'작업 폴더 준비 완료: {WORK_DIR}')

---
## SECTION 1  |  Google Drive 마운트 & 공용 드라이브 연결

### 1-1. 개인 Drive 마운트
노트북 작업 파일의 임시 저장·백업용으로 개인 Google Drive를 연결합니다.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception as e:
    IN_COLAB = False
    print('Colab 환경이 아니거나 Drive 마운트가 불가능합니다.')
    print(f'상세: {e}')

DRIVE_BASE = '/content/drive/MyDrive/VLA_2026'
os.makedirs(DRIVE_BASE, exist_ok=True)
print(f'Drive 작업 경로: {DRIVE_BASE}')

### 1-2. 공유 폴더 마운트
팀 공유 파일은 공유 폴더를 사용하여 관리합니다.

**처음 실행 시**:
* 제공된 공유 폴더 링크에 접속합니다. (https://drive.google.com/drive/folders/1sP0yHhKYPg2tGAe3GXdMIGWjvWoj3FyV)
* 폴더를 우클릭하여 `드라이브에 바로가기 추가`를 클릭합니다.
* `내 드라이브`를 선택하고 바로가기를 생성합니다.

In [ ]:
FOLDER_NAME = 'VLA_Team_2026'
SHARED_DRIVE_PATH = f'/content/drive/MyDrive/{FOLDER_NAME}'

TEAM_PATHS = {
    'notebooks': f'{SHARED_DRIVE_PATH}/notebooks/',
    'results'  : f'{SHARED_DRIVE_PATH}/results/',
    'models'   : f'{SHARED_DRIVE_PATH}/models/',
    'data'     : f'{SHARED_DRIVE_PATH}/data/',
}

print(f'설정된 경로: {SHARED_DRIVE_PATH}\n')
if os.path.exists(SHARED_DRIVE_PATH):
    for name, path in TEAM_PATHS.items():
        if os.path.exists(path):
            print(f'✓ {name:12s}: 연결 완료')
        else:
            print(f'⚠ {name:12s}: 폴더 미생성 - {path}')
else:
    print(f'❌ 오류: "{FOLDER_NAME}" 폴더를 찾을 수 없습니다.')
    print('   -> 구글 드라이브에서 "드라이브에 바로가기 추가"를 완료했는지 확인하세요.')

### 1-3. 파일 저장 편의 함수
이미지, GIF, JSON, 완성 노트북을 공유 폴더로 복사할 때 사용합니다.

In [ ]:
import shutil
from datetime import datetime

def team_save(local_path: str, folder: str = 'notebooks'):
    """로컬 파일을 공유 폴더 내 지정된 디렉토리로 복사"""
    if 'TEAM_PATHS' not in globals():
        print('❌ 오류: TEAM_PATHS가 정의되지 않았습니다. 마운트 블록을 먼저 실행하세요.')
        return None
    target_dir = TEAM_PATHS.get(folder)
    if not target_dir:
        print(f"❌ 오류: 유효하지 않은 폴더 키입니다. 선택 가능: {list(TEAM_PATHS.keys())}")
        return None
    os.makedirs(target_dir, exist_ok=True)
    filename = os.path.basename(local_path)
    target_path = os.path.join(target_dir, filename)
    try:
        shutil.copy2(local_path, target_path)
        print(f'✓ 공유 폴더 저장 완료: {target_path}')
        return target_path
    except PermissionError:
        print('❌ 권한 오류: 해당 공유 폴더에 쓰기 권한이 있는지 확인하세요.')
    except Exception as e:
        print(f'❌ 저장 실패: {e}')
    return None

def team_load(filename: str, folder: str = 'data'):
    """공유 폴더 내 파일을 Colab VM의 /tmp 디렉토리로 복사 후 경로 반환"""
    if 'TEAM_PATHS' not in globals():
        print('❌ 오류: TEAM_PATHS가 정의되지 않았습니다.')
        return None
    src = os.path.join(TEAM_PATHS.get(folder, ''), filename)
    if not os.path.exists(src):
        print(f'❌ 파일 미발견: {src}')
        return None
    dst = os.path.join('/tmp', filename)
    try:
        shutil.copy2(src, dst)
        print(f'✓ 로컬 로드 완료: {dst}')
        return dst
    except Exception as e:
        print(f'❌ 로드 실패: {e}')
        return None

print('✓ 공유 폴더 편의 함수 등록 완료')

---
## SECTION 2  |  MuJoCo 설치와 첫 렌더링

### 2-1. MuJoCo란?
* Multi-Joint Dynamics with Contact (접촉을 포함한 다관절 동역학)
* MuJoCo is a physics engine.
* MuJoCo aims to facilitate research and development in robotics and reinforcement learning.
* MuJoCo is freely available by DeepMind in October 2021.


### 2-2. MuJoCo 설치
공식 Python 바인딩은 `pip install mujoco`로 설치합니다. Colab에서 화면 없이 이미지를 렌더링하기 위해 `MUJOCO_GL=egl`을 설정합니다.

참고:
* https://github.com/google-deepmind/mujoco
* https://github.com/google-deepmind/mujoco/blob/main/python/tutorial.ipynb

In [ ]:
os.environ['MUJOCO_GL'] = 'egl'
!pip install -q mujoco mediapy imageio

import json
import math
import random
import textwrap
import time

import imageio.v2 as imageio
import matplotlib.pyplot as plt
import mediapy as media
import mujoco
import numpy as np
from IPython.display import display
from PIL import Image

print(f'✓ MuJoCo version: {mujoco.__version__}')
print(f'✓ MUJOCO_GL     : {os.environ.get("MUJOCO_GL")}')

### 2-2. MuJoCo의 세 가지 핵심 객체

| 이름 | 역할 |
|------|------|
| `MjModel` | XML을 컴파일한 모델. 몸체, 관절, 지오메트리, actuator 같은 고정 구조를 담습니다. |
| `MjData` | 시뮬레이션 중 바뀌는 상태. 위치 `qpos`, 속도 `qvel`, 제어 입력 `ctrl` 등이 들어 있습니다. |
| `Renderer` | 현재 `MjData` 상태를 카메라 이미지로 바꿉니다. VLM 입력 이미지는 여기서 나옵니다. |


* You can defines models in XML file using the MJCF language.
  * With XML file, MuJoCo constructs the models.
  * You declare a worldbody first and define a body inside it.
* geom element contains name, type, pose, and rgba.
  * type: box, sphere, cylinder and so on


In [ ]:
primitive_xml = """
<mujoco model="first_scene">
  <visual>
    <headlight diffuse="0.6 0.6 0.6" ambient="0.3 0.3 0.3" specular="0.1 0.1 0.1"/>
    <global azimuth="120" elevation="-20"/>
  </visual>
  <worldbody>
    <light name="top_light" pos="0 0 2" dir="0 0 -1" directional="true"/>
    <geom name="floor" type="plane" size="2 2 0.05" rgba="0.85 0.85 0.82 1"/>
    <geom name="red_box" type="box" pos="0 0 0.15" size="0.15 0.15 0.15" rgba="1 0.2 0.15 1"/>
    <geom name="green_sphere" type="sphere" pos="0.45 0.15 0.12" size="0.12" rgba="0.1 0.8 0.3 1"/>
    <camera name="overview" pos="1.2 -1.4 1.0" xyaxes="0.75 0.66 0 -0.34 0.39 0.86"/>
  </worldbody>
</mujoco>
"""

model = mujoco.MjModel.from_xml_string(primitive_xml)
data = mujoco.MjData(model)
print(f'geom 개수  : {model.ngeom}')
print(f'관절 자유도: {model.nv}')
print(f'qpos shape : {data.qpos.shape}')
print(f'qvel shape : {data.qvel.shape}')

In [ ]:
def render_rgb(model, data, camera=None, width=640, height=480, fallback=True):
    """MuJoCo 상태를 RGB NumPy 배열로 렌더링합니다.
    """
    mujoco.mj_forward(model, data)

    with mujoco.Renderer(model, height=height, width=width) as renderer:
        if camera is None:
            renderer.update_scene(data)
        else:
            renderer.update_scene(data, camera=camera)
        image = renderer.render()

    if fallback and camera is not None and image.max() < 5:
        print(f'⚠ camera="{camera}" 렌더가 거의 검은 화면입니다. 기본 free camera로 다시 렌더링합니다.')
        return render_rgb(model, data, camera=None, width=width, height=height, fallback=False)

    return image

def show_rgb(image, title=None, figsize=(7, 5)):
    plt.figure(figsize=figsize)
    plt.imshow(image)
    if title:
        plt.title(title)
    plt.axis('off')
    plt.show()

image = render_rgb(model, data, camera='overview')
print('렌더 이미지 shape:', image.shape, 'dtype:', image.dtype)
print('pixel min/max/mean:', image.min(), image.max(), round(float(image.mean()), 2))
show_rgb(image, 'First MuJoCo render')


### 2-3. 시뮬레이션 한 걸음 진행하기
`mj_step(model, data)`는 물리 시뮬레이션을 한 timestep 진행합니다. 지금 씬에는 움직이는 관절이 없으므로 그림은 거의 같지만, 이후 로봇 제어에서는 이 함수가 계속 사용됩니다.

In [ ]:
for _ in range(20):
    mujoco.mj_step(model, data)

image_after = render_rgb(model, data, camera='overview')
show_rgb(image_after, '20 step 이후')

---
## SECTION 3  |  MJCF로 씬 만들기

### 3-1. 우리 프로젝트용 기본 씬
MJCF는 MuJoCo에서 쓰는 XML 기반 모델 형식입니다. 오늘은 테이블, 카메라, 정리할 물체를 XML로 정의합니다.

In [ ]:
project_xml = """
<mujoco model="tabletop_objects">
  <option timestep="0.01" gravity="0 0 -9.81"/>
  <visual>
    <headlight diffuse="0.6 0.6 0.6" ambient="0.3 0.3 0.3" specular="0.1 0.1 0.1"/>
    <global azimuth="135" elevation="-25"/>
  </visual>
  <asset>
    <texture type="skybox" builtin="gradient" rgb1="0.45 0.55 0.70" rgb2="0.05 0.08 0.10" width="512" height="3072"/>
    <material name="table_mat" rgba="0.52 0.42 0.32 1"/>
  </asset>
  <worldbody>
    <light name="key_light" pos="0 -1.0 2.5" dir="0 0 -1" directional="true"/>
    <geom name="floor" type="plane" size="3 3 0.05" rgba="0.82 0.84 0.86 1"/>
    <geom name="table" type="box" pos="0.45 0 0.38" size="0.75 0.55 0.04" material="table_mat"/>
    <body name="target_box" pos="0.45 -0.15 0.48"><freejoint/><geom name="target_box_geom" type="box" size="0.07 0.07 0.05" rgba="0.1 0.35 1.0 1" mass="0.2"/></body>
    <body name="blocker_cylinder" pos="0.30 -0.02 0.50"><freejoint/><geom name="blocker_cylinder_geom" type="cylinder" size="0.06 0.07" rgba="1.0 0.28 0.1 1" mass="0.2"/></body>
    <body name="irrelevant_capsule" pos="0.70 0.22 0.51"><freejoint/><geom name="irrelevant_capsule_geom" type="capsule" size="0.035 0.11" rgba="0.2 0.8 0.35 1" mass="0.15"/></body>
    <camera name="vlm_camera" pos="1.25 -1.15 1.10" xyaxes="0.68 0.73 0 -0.40 0.37 0.84"/>
  </worldbody>
</mujoco>
"""

project_xml_path = XML_DIR / 'tabletop_objects.xml'
project_xml_path.write_text(project_xml, encoding='utf-8')
scene_model = mujoco.MjModel.from_xml_string(project_xml)
scene_data = mujoco.MjData(scene_model)
for _ in range(100):
    mujoco.mj_step(scene_model, scene_data)
scene_image = render_rgb(scene_model, scene_data, camera='vlm_camera')
scene_image_path = IMAGE_DIR / 'tabletop_objects.png'
Image.fromarray(scene_image).save(scene_image_path)
print(f'XML 저장: {project_xml_path}')
print(f'이미지 저장: {scene_image_path}')
show_rgb(scene_image, '테이블 위 물체 씬')

### 3-2. 카메라 이미지는 NumPy 배열입니다
렌더링 결과는 `(height, width, 3)` 모양의 `uint8` 배열입니다. 이 배열이 이후 CLIP, VLM, VLA 모델의 이미지 입력이 됩니다.

In [ ]:
print('shape:', scene_image.shape)
print('dtype:', scene_image.dtype)
print('min/max pixel:', scene_image.min(), scene_image.max())

plt.figure(figsize=(10, 3))
for channel_idx, channel_name in enumerate(['R', 'G', 'B']):
    plt.subplot(1, 3, channel_idx + 1)
    plt.imshow(scene_image[:, :, channel_idx], cmap='gray')
    plt.title(f'{channel_name} channel')
    plt.axis('off')
plt.tight_layout()
plt.show()

---
## SECTION 4  |  Menagerie Panda 불러오기

### 4-1. MuJoCo Menagerie란?
MuJoCo Menagerie는 여러 로봇의 MJCF 모델을 모아둔 공식 레포지터리입니다. 오늘은 `franka_emika_panda` 모델을 사용합니다.

참고:
* https://github.com/google-deepmind/mujoco_menagerie
* https://github.com/google-deepmind/mujoco_menagerie/tree/main/franka_emika_panda

In [ ]:
MENAGERIE_DIR = WORK_DIR / 'mujoco_menagerie'
if not MENAGERIE_DIR.exists():
    print('MuJoCo Menagerie clone 중...')
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/google-deepmind/mujoco_menagerie.git', str(MENAGERIE_DIR)], check=True)
else:
    print('이미 Menagerie 폴더가 있습니다. clone을 건너뜁니다.')

PANDA_DIR = MENAGERIE_DIR / 'franka_emika_panda'
PANDA_XML = PANDA_DIR / 'panda.xml'
print(f'Panda XML: {PANDA_XML}')
print('존재 여부:', PANDA_XML.exists())

### 4-2. Panda + 테이블 + 카메라 씬 만들기
Menagerie의 `panda.xml`을 include하고, 우리 프로젝트용 테이블과 카메라를 추가합니다. 이 방식은 나중에 다른 로봇으로 바꿀 때도 유용합니다.

In [ ]:
panda_project_xml = """
<mujoco model="vla_panda_tabletop">
  <include file="panda.xml"/>
  <option timestep="0.005" gravity="0 0 -9.81"/>
  <statistic center="0.45 0 0.45" extent="1.2"/>
  <visual>
    <headlight diffuse="0.6 0.6 0.6" ambient="0.3 0.3 0.3" specular="0.1 0.1 0.1"/>
    <global azimuth="135" elevation="-22"/>
  </visual>
  <asset>
    <texture type="skybox" builtin="gradient" rgb1="0.35 0.45 0.60" rgb2="0.02 0.03 0.04" width="512" height="3072"/>
    <material name="table_mat" rgba="0.50 0.41 0.33 1"/>
  </asset>
  <worldbody>
    <light name="key_light" pos="0 -1.0 2.5" dir="0 0 -1" directional="true"/>
    <geom name="floor" type="plane" size="3 3 0.05" rgba="0.82 0.84 0.86 1"/>
    <geom name="table" type="box" pos="0.55 0 0.15" size="0.65 0.50 0.04" material="table_mat" contype="1" conaffinity="1"/>
    <body name="target_box" pos="0.50 -0.14 0.43"><freejoint/><geom name="target_box_geom" type="box" size="0.045 0.045 0.04" rgba="0.1 0.35 1.0 1" mass="0.15"/></body>
    <body name="blocker_cylinder" pos="0.38 -0.03 0.44"><freejoint/><geom name="blocker_cylinder_geom" type="cylinder" size="0.045 0.055" rgba="1.0 0.28 0.1 1" mass="0.15"/></body>
    <body name="irrelevant_capsule" pos="0.70 0.22 0.45"><freejoint/><geom name="irrelevant_capsule_geom" type="capsule" size="0.025 0.09" rgba="0.2 0.8 0.35 1" mass="0.12"/></body>
    <camera name="vlm_camera" pos="1.25 -1.15 1.05" xyaxes="0.68 0.73 0 -0.39 0.36 0.85"/>
    <camera name="side_camera" pos="0.0 -1.6 0.75" xyaxes="1 0 0 0 0.35 0.94"/>
  </worldbody>
</mujoco>
"""

panda_scene_path = PANDA_DIR / 'vla_panda_tabletop.xml'
panda_scene_path.write_text(panda_project_xml, encoding='utf-8')
panda_model = mujoco.MjModel.from_xml_path(str(panda_scene_path))
panda_data = mujoco.MjData(panda_model)

print(f'Panda scene XML 저장: {panda_scene_path}')
print(f'nq={panda_model.nq}, nv={panda_model.nv}, nu={panda_model.nu}, ngeom={panda_model.ngeom}')
print('actuator names:')
for i in range(panda_model.nu):
    print(f'  {i}: {mujoco.mj_id2name(panda_model, mujoco.mjtObj.mjOBJ_ACTUATOR, i)}')

In [ ]:
panda_model = mujoco.MjModel.from_xml_path(str(panda_scene_path))

# Panda root body를 테이블 위로 올리기
body_id = mujoco.mj_name2id(
    panda_model,
    mujoco.mjtObj.mjOBJ_BODY,
    'link0'   # Menagerie Panda의 베이스 body 이름
)

panda_model.body_pos[body_id] = [0.25, 0.0, 0.20]

panda_data = mujoco.MjData(panda_model)

In [ ]:
if panda_model.nkey > 0:
    mujoco.mj_resetDataKeyframe(panda_model, panda_data, 0)
else:
    mujoco.mj_resetData(panda_model, panda_data)
for _ in range(100):
    mujoco.mj_step(panda_model, panda_data)

panda_image = render_rgb(panda_model, panda_data, camera='vlm_camera')
panda_image_path = IMAGE_DIR / 'panda_tabletop.png'
Image.fromarray(panda_image).save(panda_image_path)
print(f'이미지 저장: {panda_image_path}')
show_rgb(panda_image, 'Panda + 정리 물체 씬')

---
## SECTION 5  |  Panda 관절과 그리퍼 제어

### 5-1. 제어 입력 `data.ctrl`
Menagerie Panda 모델은 8개의 actuator를 사용합니다.

| actuator | 의미 |
|----------|------|
| `actuator1` ~ `actuator7` | Panda arm 7개 관절의 목표값 |
| `actuator8` | 그리퍼 열림/닫힘 제어 |

In [ ]:
def reset_panda(model, data):
    if model.nkey > 0:
        mujoco.mj_resetDataKeyframe(model, data, 0)
    else:
        mujoco.mj_resetData(model, data)
    mujoco.mj_forward(model, data)

PANDA_POSES = {
    'home_open':  [0.0, -0.55, 0.0, -2.20, 0.0, 1.75, 0.78, 255.0],
    'look_left':  [0.45, -0.65, 0.10, -2.10, 0.10, 1.65, 1.00, 255.0],
    'look_right': [-0.45, -0.65, -0.10, -2.10, -0.10, 1.65, 0.55, 255.0],
    'reach_low':  [0.10, -0.35, 0.10, -2.45, 0.00, 2.05, 0.85, 255.0],
    'close_grip': [0.10, -0.35, 0.10, -2.45, 0.00, 2.05, 0.85, 0.0],
}

def apply_pose(model, data, pose, steps=180):
    data.ctrl[:] = np.array(pose, dtype=np.float64)
    frames = []
    for step in range(steps):
        mujoco.mj_step(model, data)
        if step % 15 == 0:
            frames.append(render_rgb(model, data, camera='vlm_camera', width=480, height=360))
    return frames

reset_panda(panda_model, panda_data)
frames = apply_pose(panda_model, panda_data, PANDA_POSES['reach_low'])
show_rgb(frames[-1], 'Panda reach_low pose')

### 5-2. 여러 자세를 GIF로 확인하기
아래 셀은 preset pose를 차례로 적용하고 짧은 GIF를 만듭니다. GIF는 발표 자료나 진행 기록에 넣기 좋습니다.

In [ ]:
reset_panda(panda_model, panda_data)
all_frames = []
for pose_name in ['home_open', 'look_left', 'look_right', 'reach_low', 'close_grip']:
    print(f'pose 실행: {pose_name}')
    all_frames.extend(apply_pose(panda_model, panda_data, PANDA_POSES[pose_name], steps=120))

panda_gif_path = GIF_DIR / 'panda_pose_demo.gif'
imageio.mimsave(panda_gif_path, all_frames, fps=8)
print(f'GIF 저장: {panda_gif_path}')
media.show_video(all_frames, fps=8)

### 5-3. 관절 상태 기록하기
VLA 학습 데이터에는 로봇의 상태와 제어 입력이 들어갑니다. 여기서는 현재 `qpos`, `qvel`, `ctrl`의 일부를 확인합니다.

---
## SECTION 6  |  기숙사 로비 택배 정리 환경

### 6-1. 로비 바닥 위 택배 씬 생성 아이디어
이 프로젝트의 실제 상황은 **테이블 위 물체 정리**가 아니라 **기숙사 로비 바닥에 놓인 택배를 정리하는 것**입니다. 따라서 이 섹션부터는 테이블을 제거하고, 물체를 모두 바닥 위에 놓습니다.

| 요소 | 역할 |
|------|------|
| robot | 로비 바닥에 놓인 Panda 로봇 팔. 실제 프로젝트에서는 이동형 베이스나 다른 로봇으로 바뀔 수 있습니다. |
| target_package | 정리해야 하는 택배 |
| blocker_package | target으로 가는 경로나 주변을 막는 택배 |
| irrelevant_package | 지금 명령과 상관없는 택배 |
| vlm_camera | VLM이 볼 로비 장면 카메라 |

`action_label`은 완벽한 정답 생성기가 아니라, 씬을 해석하는 연습을 위한 간단한 규칙입니다.

In [ ]:
PACKAGE_SPECS = {
    'small_box': {
        'type': 'box',
        'size_fn': lambda rng: [round(rng.uniform(0.055, 0.090), 3), round(rng.uniform(0.045, 0.075), 3), round(rng.uniform(0.035, 0.060), 3)],
    },
    'wide_box': {
        'type': 'box',
        'size_fn': lambda rng: [round(rng.uniform(0.090, 0.140), 3), round(rng.uniform(0.050, 0.085), 3), round(rng.uniform(0.035, 0.065), 3)],
    },
    'tube_package': {
        'type': 'cylinder',
        'size_fn': lambda rng: [round(rng.uniform(0.035, 0.055), 3), round(rng.uniform(0.080, 0.140), 3)],
    },
}

PACKAGE_COLORS = {
    'target': [0.95, 0.70, 0.34, 1.0],      # cardboard
    'blocker': [0.78, 0.45, 0.22, 1.0],
    'irrelevant': [0.55, 0.55, 0.50, 1.0],
}

DEFAULT_ROBOT_BASE_POS = [0.0, 0.0, 0.0]
DEFAULT_ROBOT_BASE_YAW = 0.0
DEFAULT_CAMERA_POS = [1.75, -1.60, 1.15]
DEFAULT_CAMERA_LOOKAT = [0.70, 0.00, 0.12]
DEFAULT_LIGHT_POS = [0.0, -1.0, 2.8]
DEFAULT_PACKAGE_SCALES = {'small_box': 1.0, 'wide_box': 1.0, 'tube_package': 1.0}

def camera_xyaxes_from_pos_lookat(pos, lookat, up=(0, 0, 1)):
    """MuJoCo fixed camera의 xyaxes를 pos/lookat에서 계산합니다."""
    pos = np.array(pos, dtype=float)
    lookat = np.array(lookat, dtype=float)
    up = np.array(up, dtype=float)

    forward = lookat - pos
    forward = forward / np.linalg.norm(forward)
    z_axis = -forward
    x_axis = np.cross(up, z_axis)
    if np.linalg.norm(x_axis) < 1e-6:
        x_axis = np.array([1.0, 0.0, 0.0])
    x_axis = x_axis / np.linalg.norm(x_axis)
    y_axis = np.cross(z_axis, x_axis)
    y_axis = y_axis / np.linalg.norm(y_axis)
    return ' '.join(f'{v:.6f}' for v in np.r_[x_axis, y_axis])

def package_z(shape_name, size):
    if PACKAGE_SPECS[shape_name]['type'] == 'box':
        return size[2]
    if PACKAGE_SPECS[shape_name]['type'] == 'cylinder':
        return size[1]
    return size[-1]

def choose_action_label(target_xy, blocker_xy, irrelevant_xy):
    target_blocker_dist = float(np.linalg.norm(np.array(target_xy) - np.array(blocker_xy)))
    if target_blocker_dist < 0.22:
        return 'push'
    if abs(target_xy[1]) > 0.35:
        return 'reorder'
    if irrelevant_xy[0] < 0.35:
        return 'ignore'
    return 'pick'

def package_body(name, geom_type, size, pos, rgba, mass=0.20, friction=0.8):
    size_str = ' '.join(map(str, size))
    pos_str = ' '.join(map(str, pos))
    rgba_str = ' '.join(map(str, rgba))
    return '\n'.join([
        f'    <body name="{name}" pos="{pos_str}">',
        '      <freejoint/>',
        f'      <geom name="{name}_geom" type="{geom_type}" size="{size_str}" rgba="{rgba_str}" mass="{mass:.3f}" friction="{friction:.3f} 0.02 0.001"/>',
        '    </body>',
    ])

def set_panda_base_pose(model, base_pos=DEFAULT_ROBOT_BASE_POS, base_yaw=DEFAULT_ROBOT_BASE_YAW):
    """include된 Panda root body를 원하는 로비 좌표로 옮깁니다."""
    for body_name in ['link0', 'panda_link0']:
        body_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, body_name)
        if body_id >= 0:
            model.body_pos[body_id] = np.array(base_pos, dtype=float)
            half = float(base_yaw) / 2.0
            model.body_quat[body_id] = np.array([math.cos(half), 0.0, 0.0, math.sin(half)])
            return body_name
    print('⚠ Panda root body(link0)를 찾지 못했습니다. body 이름을 출력해 확인하세요.')
    return None

def generate_lobby_scene_from_layout(
    scene_id='lobby_scene_custom',
    target_xy=(0.75, -0.12),
    blocker_xy=(0.58, -0.03),
    irrelevant_xy=(1.05, 0.30),
    camera_pos=DEFAULT_CAMERA_POS,
    camera_lookat=DEFAULT_CAMERA_LOOKAT,
    robot_base_pos=DEFAULT_ROBOT_BASE_POS,
    robot_base_yaw=DEFAULT_ROBOT_BASE_YAW,
    package_shapes=None,
    package_scales=None,
    package_mass=0.20,
    package_friction=0.80,
    light_pos=DEFAULT_LIGHT_POS,
    light_diffuse=0.65,
    light_ambient=0.35,
    floor_reflectance=0.08,
    seed=0,
):
    rng = random.Random(seed)
    package_scales = package_scales or DEFAULT_PACKAGE_SCALES
    package_shapes = package_shapes or {}
    roles = [
        ('target', 'target_package', list(target_xy)),
        ('blocker', 'blocker_package', list(blocker_xy)),
        ('irrelevant', 'irrelevant_package', list(irrelevant_xy)),
    ]

    bodies, objects_meta = [], []
    for role, name, xy in roles:
        shape_name = package_shapes.get(role) or rng.choice(list(PACKAGE_SPECS.keys()))
        spec = PACKAGE_SPECS[shape_name]
        size = spec['size_fn'](rng)
        scale = float(package_scales.get(shape_name, 1.0))
        size = [round(v * scale, 3) for v in size]
        pos = [round(xy[0], 3), round(xy[1], 3), round(package_z(shape_name, size), 3)]
        rgba = PACKAGE_COLORS[role]
        bodies.append(package_body(name, spec['type'], size, pos, rgba, mass=package_mass, friction=package_friction))
        objects_meta.append({'name': name, 'role': role, 'shape': shape_name, 'position': pos, 'size': size, 'rgba': rgba})

    action_label = choose_action_label(target_xy, blocker_xy, irrelevant_xy)
    xyaxes = camera_xyaxes_from_pos_lookat(camera_pos, camera_lookat)
    camera_pos_str = ' '.join(f'{v:.3f}' for v in camera_pos)
    light_pos_str = ' '.join(f'{v:.3f}' for v in light_pos)
    bodies_xml = '\n'.join(bodies)

    xml = f"""
<mujoco model="{scene_id}">
  <include file="panda.xml"/>
  <option timestep="0.005" gravity="0 0 -9.81"/>
  <statistic center="0.65 0 0.35" extent="1.6"/>
  <visual>
    <headlight diffuse="{light_diffuse:.3f} {light_diffuse:.3f} {light_diffuse:.3f}" ambient="{light_ambient:.3f} {light_ambient:.3f} {light_ambient:.3f}" specular="0.1 0.1 0.1"/>
    <global azimuth="135" elevation="-18"/>
  </visual>
  <asset>
    <texture type="skybox" builtin="gradient" rgb1="0.45 0.52 0.60" rgb2="0.04 0.05 0.06" width="512" height="3072"/>
    <texture type="2d" name="lobby_tile" builtin="checker" rgb1="0.78 0.78 0.74" rgb2="0.68 0.68 0.64" width="128" height="128"/>
    <material name="lobby_floor" texture="lobby_tile" texrepeat="8 8" texuniform="true" reflectance="{floor_reflectance:.3f}"/>
  </asset>
  <worldbody>
    <light name="lobby_light" pos="{light_pos_str}" dir="0 0 -1" directional="true"/>
    <geom name="lobby_floor" type="plane" size="3.0 3.0 0.05" material="lobby_floor"/>
{bodies_xml}
    <camera name="vlm_camera" pos="{camera_pos_str}" xyaxes="{xyaxes}"/>
  </worldbody>
</mujoco>
"""
    metadata = {
        'scene_id': scene_id,
        'environment': 'dorm_lobby_floor',
        'camera': 'vlm_camera',
        'camera_pos': list(camera_pos),
        'camera_lookat': list(camera_lookat),
        'robot_base_pos': list(robot_base_pos),
        'robot_base_yaw': float(robot_base_yaw),
        'package_shapes': dict(package_shapes),
        'package_scales': dict(package_scales),
        'package_mass': float(package_mass),
        'package_friction': float(package_friction),
        'light_pos': list(light_pos),
        'light_diffuse': float(light_diffuse),
        'light_ambient': float(light_ambient),
        'floor_reflectance': float(floor_reflectance),
        'objects': objects_meta,
        'lang_instruction': '기숙사 로비의 택배를 정리하시오.',
        'action_label': action_label,
        'notes': 'floor-based dorm lobby parcel scene; rule-based label for tutorial only',
    }
    return xml, metadata

def generate_random_scene(seed=0):
    rng = random.Random(seed)
    target_xy = [round(rng.uniform(0.55, 1.05), 3), round(rng.uniform(-0.38, 0.28), 3)]
    blocker_xy = [round(target_xy[0] + rng.uniform(-0.22, 0.16), 3), round(target_xy[1] + rng.uniform(-0.18, 0.18), 3)]
    irrelevant_xy = [round(rng.uniform(0.45, 1.25), 3), round(rng.uniform(0.25, 0.55), 3)]
    return generate_lobby_scene_from_layout(
        scene_id=f'lobby_scene_{seed:03d}',
        target_xy=target_xy,
        blocker_xy=blocker_xy,
        irrelevant_xy=irrelevant_xy,
        package_shapes={'target': 'wide_box', 'blocker': 'small_box', 'irrelevant': 'tube_package'},
        seed=seed,
    )

LOBBY_PANDA_HOME_QPOS = {
    'joint1': 0.0,
    'joint2': -0.55,
    'joint3': 0.0,
    'joint4': -2.20,
    'joint5': 0.0,
    'joint6': 1.75,
    'joint7': 0.78,
    'finger_joint1': 0.04,
    'finger_joint2': 0.04,
}

def initialize_lobby_data(model, data):
    """로비 씬 초기화.

    Panda keyframe으로 전체 qpos를 리셋하면 새로 추가한 택배 freejoint 위치까지
    덮어쓸 수 있으므로, 로비 씬에서는 mj_resetData 후 Panda 관절만 직접 설정합니다.
    이렇게 해야 택배가 XML에 정의된 바닥 위치에서 시작합니다.
    """
    mujoco.mj_resetData(model, data)

    for joint_name, qpos_value in LOBBY_PANDA_HOME_QPOS.items():
        joint_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, joint_name)
        if joint_id >= 0:
            qpos_addr = model.jnt_qposadr[joint_id]
            data.qpos[qpos_addr] = qpos_value

    if model.nu >= 8:
        data.ctrl[:] = np.array([0.0, -0.55, 0.0, -2.20, 0.0, 1.75, 0.78, 255.0], dtype=np.float64)

    mujoco.mj_forward(model, data)

def load_lobby_model(xml_path, metadata):
    model = mujoco.MjModel.from_xml_path(str(xml_path))
    set_panda_base_pose(
        model,
        base_pos=metadata.get('robot_base_pos', DEFAULT_ROBOT_BASE_POS),
        base_yaw=metadata.get('robot_base_yaw', DEFAULT_ROBOT_BASE_YAW),
    )
    data = mujoco.MjData(model)
    initialize_lobby_data(model, data)
    return model, data


### 6-2. 왜 박스가 안 보일 수 있나요? 물리는 적용되나요?

`PACKAGE_SPECS`는 가능한 택배 모양 후보를 정의합니다. 기존 코드는 각 물체마다 `rng.choice(...)`로 모양을 골랐기 때문에, 특정 seed에서는 원통형 `tube_package`가 여러 개 선택될 수 있습니다. 예를 들어 이전 `seed=42`에서는 target과 irrelevant가 원통이고, blocker만 작은 박스라서 카메라 각도나 로봇 베이스에 가려 박스가 잘 안 보일 수 있었습니다.

본 예제의 기본 랜덤 씬은 `target=wide_box`, `blocker=small_box`, `irrelevant=tube_package`로 고정해 세 종류가 모두 보이게 했습니다. GUI에서는 각 물체의 shape 드롭다운을 바꾸면 됩니다.

MuJoCo에 적용되는 물리적 특성 :

* `<option gravity="0 0 -9.81"/>`: 지구 중력과 같은 방향의 중력이 적용됩니다.
* 각 택배 body에는 `<freejoint/>`가 있어서 바닥 위에서 자유롭게 움직이고 회전할 수 있습니다.
* 각 geom에는 `mass`와 `friction`이 들어갑니다. `friction="0.8 0.02 0.001"`의 첫 값은 미끄럼 마찰, 뒤 값들은 회전/비틀림 계열 마찰입니다.
* 렌더링 전에 `mj_step`을 여러 번 실행하기 때문에, 물체는 중력과 접촉 힘에 의해 바닥에 안정화됩니다.

주의: 현재 코드는 “정교한 택배 조작 시뮬레이터”가 아니라 VLM/VLA 데이터 생성 개념을 보여주는 튜토리얼입니다. 실제 밀기/집기 실험에서는 물체 크기, 질량, 마찰, 로봇 end-effector 위치, 접촉 안정성을 더 엄격히 튜닝해야 합니다.

중요한 초기화 주의점: Menagerie Panda에는 keyframe이 들어 있습니다. `mj_resetDataKeyframe()`으로 전체 상태를 리셋하면 Panda 관절뿐 아니라 이후 추가한 택배 freejoint 위치까지 keyframe 값으로 덮일 수 있습니다. 그래서 로비 씬에서는 `initialize_lobby_data()`가 `mj_resetData()`로 택배의 XML 초기 위치를 보존한 뒤, Panda 관절값만 따로 설정합니다. 이 처리가 없으면 물체가 바닥에서 안정화된 상태가 아니라 떨어지는 중처럼 보일 수 있습니다.


### 6-2. 랜덤 로비 씬 하나 렌더링하기
아래 셀을 여러 번 실행하거나 `seed` 값을 바꾸면 로비 바닥의 택배 배치가 달라집니다.

In [ ]:
seed = 42
random_xml, random_meta = generate_random_scene(seed=seed)
random_scene_path = PANDA_DIR / f'{random_meta["scene_id"]}.xml'
random_scene_path.write_text(random_xml, encoding='utf-8')

random_model, random_data = load_lobby_model(random_scene_path, random_meta)
random_data.ctrl[:] = np.array(PANDA_POSES['home_open'], dtype=np.float64)
for _ in range(120):
    mujoco.mj_step(random_model, random_data)

random_image = render_rgb(random_model, random_data, camera='vlm_camera')
random_image_path = IMAGE_DIR / f'{random_meta["scene_id"]}.png'
Image.fromarray(random_image).save(random_image_path)
random_meta['image_path'] = str(random_image_path)
random_meta['robot_qpos'] = random_data.qpos.tolist()
print(json.dumps(random_meta, ensure_ascii=False, indent=2)[:1600])
show_rgb(random_image, f'{random_meta["scene_id"]} | label={random_meta["action_label"]}')

### 6-3. Colab에서 배치를 직접 조정하기

MuJoCo의 데스크톱 viewer처럼 마우스로 물체를 끌어다 놓는 GUI는 Colab 브라우저 안에서 안정적으로 쓰기 어렵습니다.
대신 Colab에서는 `ipywidgets` 슬라이더와 버튼으로 다음 값을 조정하는 방식을 사용합니다.

* 로봇 베이스 위치 `robot_x`, `robot_y`
* target/blocker/irrelevant 택배 위치
* VLM 카메라 위치와 바라보는 지점

더 정교한 편집이 필요하면 로컬 PC에서 `python -m mujoco.viewer` 또는 별도 Python 스크립트로 XML을 확인한 뒤, 수정된 XML을 Drive/GitHub에 올려 Colab에서 다시 불러오는 방식을 사용합니다.

In [ ]:
try:
    import ipywidgets as widgets
except ImportError:
    !pip install -q ipywidgets
    import ipywidgets as widgets

from io import BytesIO

slider_style = {'description_width': '120px'}
slider_layout = widgets.Layout(width='420px')

def slider(name, value, min_value, max_value, step=0.01):
    return widgets.FloatSlider(
        value=value,
        min=min_value,
        max=max_value,
        step=step,
        description=name,
        style=slider_style,
        layout=slider_layout,
        continuous_update=False,
    )

robot_x = slider('robot x', 0.00, -0.40, 0.40)
robot_y = slider('robot y', 0.00, -0.50, 0.50)
target_x = slider('target x', 0.75, 0.35, 1.35)
target_y = slider('target y', -0.12, -0.60, 0.60)
target_shape = widgets.Dropdown(options=list(PACKAGE_SPECS.keys()), value='wide_box', description='target shape', style=slider_style, layout=slider_layout)
blocker_x = slider('blocker x', 0.58, 0.35, 1.35)
blocker_y = slider('blocker y', -0.03, -0.60, 0.60)
blocker_shape = widgets.Dropdown(options=list(PACKAGE_SPECS.keys()), value='small_box', description='blocker shape', style=slider_style, layout=slider_layout)
irrelevant_x = slider('irrelevant x', 1.05, 0.35, 1.35)
irrelevant_y = slider('irrelevant y', 0.30, -0.60, 0.60)
irrelevant_shape = widgets.Dropdown(options=list(PACKAGE_SPECS.keys()), value='tube_package', description='irrelevant shape', style=slider_style, layout=slider_layout)
camera_x = slider('camera x', 1.75, -0.50, 2.50)
camera_y = slider('camera y', -1.60, -2.50, 0.50)
camera_z = slider('camera z', 1.15, 0.30, 2.20)
lookat_x = slider('lookat x', 0.70, 0.20, 1.30)
lookat_y = slider('lookat y', 0.00, -0.60, 0.60)
lookat_z = slider('lookat z', 0.12, 0.00, 0.80)
light_x = slider('light x', 0.00, -2.00, 2.00)
light_y = slider('light y', -1.00, -3.00, 1.00)
light_z = slider('light z', 2.80, 0.80, 5.00)
light_diffuse = slider('light diffuse', 0.65, 0.10, 1.20, step=0.05)
light_ambient = slider('light ambient', 0.35, 0.00, 1.00, step=0.05)
floor_reflectance = slider('floor reflect', 0.08, 0.00, 0.50, step=0.01)
small_box_scale = slider('small box scale', 1.00, 0.50, 2.00, step=0.05)
wide_box_scale = slider('wide box scale', 1.00, 0.50, 2.00, step=0.05)
tube_package_scale = slider('tube scale', 1.00, 0.50, 2.00, step=0.05)
package_mass = slider('package mass', 0.20, 0.05, 1.00, step=0.05)
package_friction = slider('package friction', 0.80, 0.10, 2.00, step=0.05)

render_button = widgets.Button(description='Render custom lobby scene', button_style='primary')
save_button = widgets.Button(description='Save current layout JSON', button_style='success')
preview = widgets.Image(format='png', width=640, height=480)
status = widgets.HTML(value='')

last_custom = {}

def update_preview(image):
    buffer = BytesIO()
    Image.fromarray(image).save(buffer, format='PNG')
    preview.value = buffer.getvalue()

def save_scene_record(metadata, data, json_dir=JSON_DIR):
    """SECTION 7을 실행하지 않아도 GUI 셀에서 JSON 저장이 가능하도록 둔 로컬 저장 함수입니다."""
    record = dict(metadata)
    record['robot_qpos'] = data.qpos.tolist()
    record['robot_qvel'] = data.qvel.tolist()
    record['created_at'] = datetime.now().isoformat(timespec='seconds')

    json_path = json_dir / f'{record["scene_id"]}.json'
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(record, f, ensure_ascii=False, indent=2)
    return json_path, record

def render_custom_scene(_=None):
    status.value = '<span style="color:#555">렌더링 중...</span>'
    xml, meta = generate_lobby_scene_from_layout(
        scene_id='lobby_scene_custom',
        target_xy=(target_x.value, target_y.value),
        blocker_xy=(blocker_x.value, blocker_y.value),
        irrelevant_xy=(irrelevant_x.value, irrelevant_y.value),
        camera_pos=[camera_x.value, camera_y.value, camera_z.value],
        camera_lookat=[lookat_x.value, lookat_y.value, lookat_z.value],
        robot_base_pos=[robot_x.value, robot_y.value, 0.0],
        package_shapes={
            'target': target_shape.value,
            'blocker': blocker_shape.value,
            'irrelevant': irrelevant_shape.value,
        },
        package_scales={
            'small_box': small_box_scale.value,
            'wide_box': wide_box_scale.value,
            'tube_package': tube_package_scale.value,
        },
        package_mass=package_mass.value,
        package_friction=package_friction.value,
        light_pos=[light_x.value, light_y.value, light_z.value],
        light_diffuse=light_diffuse.value,
        light_ambient=light_ambient.value,
        floor_reflectance=floor_reflectance.value,
        seed=999,
    )
    scene_path = PANDA_DIR / 'lobby_scene_custom.xml'
    scene_path.write_text(xml, encoding='utf-8')

    model, data = load_lobby_model(scene_path, meta)
    data.ctrl[:] = np.array(PANDA_POSES['home_open'], dtype=np.float64)
    for _ in range(100):
        mujoco.mj_step(model, data)

    image = render_rgb(model, data, camera='vlm_camera')
    image_path = IMAGE_DIR / 'lobby_scene_custom.png'
    Image.fromarray(image).save(image_path)

    meta['image_path'] = str(image_path)
    meta['robot_qpos'] = data.qpos.tolist()
    last_custom['meta'] = meta
    last_custom['data'] = data
    last_custom['image'] = image

    update_preview(image)
    status.value = f'<b>label={meta["action_label"]}</b> &nbsp; image={image_path}'

def save_custom_scene(_=None):
    if 'meta' not in last_custom:
        status.value = '<span style="color:#b00020">먼저 Render custom lobby scene 버튼을 눌러 장면을 렌더링하세요.</span>'
        return

    json_path, record = save_scene_record(last_custom['meta'], last_custom['data'])
    status.value = f'<b>custom layout JSON 저장:</b> {json_path}<br><pre>{json.dumps(record, ensure_ascii=False, indent=2)[:1600]}</pre>'

render_button.on_click(render_custom_scene)
save_button.on_click(save_custom_scene)

controls = widgets.VBox([
    widgets.HTML('<b>Robot base</b>'), robot_x, robot_y,
    widgets.HTML('<b>Packages</b>'), target_x, target_y, target_shape, blocker_x, blocker_y, blocker_shape, irrelevant_x, irrelevant_y, irrelevant_shape,
    widgets.HTML('<b>Camera</b>'), camera_x, camera_y, camera_z, lookat_x, lookat_y, lookat_z,
    widgets.HTML('<b>Lighting</b>'), light_x, light_y, light_z, light_diffuse, light_ambient, floor_reflectance,
    widgets.HTML('<b>Package properties</b>'), small_box_scale, wide_box_scale, tube_package_scale, package_mass, package_friction,
    widgets.HBox([render_button, save_button]),
])

display(controls, preview, status)
render_custom_scene()

---
## SECTION 7  |  VLM/VLA 연결을 위한 데이터 형태

### 7-1. 한 장면을 JSON으로 저장하기
VLM/VLA 연동을 시작하면 이미지 파일만으로는 부족합니다. 어떤 물체가 어디에 있었는지, 어떤 카메라로 찍었는지, 로봇 상태가 어땠는지 함께 기록해야 합니다.

In [ ]:
def save_scene_json(metadata, data, json_dir=JSON_DIR):
    record = dict(metadata)
    record['robot_qpos'] = data.qpos.tolist()
    record['robot_qvel'] = data.qvel.tolist()
    record['created_at'] = datetime.now().isoformat(timespec='seconds')
    json_path = json_dir / f'{record["scene_id"]}.json'
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(record, f, ensure_ascii=False, indent=2)
    return json_path, record

json_path, scene_record = save_scene_json(random_meta, random_data)
print(f'JSON 저장: {json_path}')
print(json.dumps(scene_record, ensure_ascii=False, indent=2)[:1800])

### 7-2. Mock VLM/VLA 파이프라인
아직 실제 VLM이나 VLA 모델을 호출하지는 않습니다. 대신 앞으로 연결될 입출력 구조를 함수로 확인합니다.

In [ ]:
def mock_vlm_planner(scene_json):
    return {'selected_action': scene_json['action_label'], 'reason': '튜토리얼에서는 물체 거리와 배치 규칙으로 생성한 라벨을 사용합니다.', 'input_image': scene_json['image_path'], 'instruction': scene_json['lang_instruction']}

def mock_vla_controller(planner_output, robot_qpos):
    action = planner_output['selected_action']
    if action == 'pick':
        target_pose = PANDA_POSES['reach_low']
    elif action == 'push':
        target_pose = PANDA_POSES['look_left']
    elif action == 'reorder':
        target_pose = PANDA_POSES['look_right']
    else:
        target_pose = PANDA_POSES['home_open']
    return {'action': action, 'target_ctrl': target_pose, 'current_qpos_head': robot_qpos[:8]}

planner_output = mock_vlm_planner(scene_record)
controller_output = mock_vla_controller(planner_output, scene_record['robot_qpos'])
print('[Mock VLM output]')
print(json.dumps(planner_output, ensure_ascii=False, indent=2))
print('\n[Mock VLA output]')
print(json.dumps(controller_output, ensure_ascii=False, indent=2))

---
## SECTION 8  |  미니 체크

### 8-1. 랜덤 씬 5개 생성·렌더링·저장
아래 셀을 실행하면 랜덤 씬 5개를 만들고, 각 씬의 이미지와 JSON을 저장합니다.

In [ ]:
def build_and_render_scene(seed):
    xml, metadata = generate_random_scene(seed=seed)
    scene_path = PANDA_DIR / f'{metadata["scene_id"]}.xml'
    scene_path.write_text(xml, encoding='utf-8')
    model, data = load_lobby_model(scene_path, metadata)
    data.ctrl[:] = np.array(PANDA_POSES['home_open'], dtype=np.float64)
    for _ in range(140):
        mujoco.mj_step(model, data)
    image = render_rgb(model, data, camera='vlm_camera')
    image_path = IMAGE_DIR / f'{metadata["scene_id"]}.png'
    Image.fromarray(image).save(image_path)
    metadata['image_path'] = str(image_path)
    json_path, record = save_scene_json(metadata, data)
    return image, image_path, json_path, record

mini_results = []
for seed in range(5):
    image, image_path, json_path, record = build_and_render_scene(seed)
    mini_results.append((image, image_path, json_path, record))
    print(f'✓ {record["scene_id"]}: label={record["action_label"]}, image={image_path.name}, json={json_path.name}')

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, (image, image_path, json_path, record) in zip(axes, mini_results):
    ax.imshow(image)
    ax.set_title(f'{record["scene_id"]}\n{record["action_label"]}')
    ax.axis('off')
plt.tight_layout()
plt.show()

### 8-2. 공유 폴더에 결과 저장하기
공유 폴더가 연결되어 있다면 이미지와 JSON을 팀 폴더에 복사합니다.

In [ ]:
for _, image_path, json_path, record in mini_results:
    team_save(str(image_path), folder='results')
    team_save(str(json_path), folder='data')
team_save(str(panda_gif_path), folder='results')

### 8-3. 제출용 메모
아래 항목을 확인한 뒤, 완성된 노트북을 공유 폴더 `notebooks/`에 저장하세요.

* `panda_pose_demo.gif`가 생성되었는가?
* `lobby_scene_000.png` ~ `lobby_scene_004.png`가 생성되었는가?
* 각 JSON에 `scene_id`, `image_path`, `camera`, `objects`, `lang_instruction`, `action_label`, `robot_qpos`, `notes`가 있는가?

In [ ]:
required_keys = {'scene_id', 'image_path', 'camera', 'objects', 'lang_instruction', 'action_label', 'robot_qpos', 'notes'}
for _, image_path, json_path, _ in mini_results:
    with open(json_path, encoding='utf-8') as f:
        record = json.load(f)
    missing = required_keys - set(record.keys())
    if missing:
        print(f'⚠ {json_path.name}: 누락 키 {missing}')
    else:
        print(f'✓ {json_path.name}: 필수 키 확인 완료')

---
## SECTION 9  |  오늘 배운 것 정리

### 오늘 다룬 내용
* `MjModel`: XML에서 컴파일된 로봇·물체·카메라 구조
* `MjData`: 현재 시뮬레이션 상태와 제어 입력
* `mj_step`: 물리 시뮬레이션을 한 timestep 진행
* `Renderer`: 시뮬레이션 상태를 RGB 이미지로 변환
* `MJCF/XML`: MuJoCo 씬을 정의하는 파일 형식
* `mujoco_menagerie`: 공식 로봇 모델 모음
* `Panda actuator`: 7개 arm actuator + 1개 gripper actuator

### 다음 단계와 연결
다음 회차에서는 오늘 만든 구조를 확장해 더 많은 씬을 자동 생성합니다.

* 물체 위치·색상·크기·조명·카메라 각도 랜덤화
* 가림 시나리오 5종 생성
* `(RGB image, object metadata, action label)` 데이터셋 저장
* scripted policy로 trajectory 수집

오늘 만든 JSON 구조가 이후 VLM/VLA 실험 데이터의 출발점입니다.

---
## SECTION 10  |  수업 후 혼자 해보기: Windows GUI Scene Editor

공식 MuJoCo viewer는 장면을 확인하고 카메라를 움직이며 시뮬레이션을 재생하는 데 좋습니다. 하지만 Blender처럼 물체를 추가하고 배치한 뒤 XML로 저장하는 완성형 scene editor는 아닙니다.

그래서 이 수업에서는 별도의 간단한 GUI 편집기 `scene_editor.py`를 사용합니다. 이 파일은 Streamlit으로 만든 로컬 웹 GUI입니다.

권장 workflow:

1. Colab에서 기본 구조를 이해한다.
2. Windows 로컬 PC에서 `scene_editor.py`로 로비 택배 씬을 GUI로 만든다.
3. 저장된 `lobby_scene_custom.xml`, `lobby_scene_custom.json`을 MuJoCo viewer로 한 번 검수한다.
4. XML/JSON을 Google Drive 또는 GitHub에 올린다.
5. Colab에서 다시 불러와 렌더링, 데이터셋 생성, VLM/VLA 연결에 사용한다.

### 10-1. 이 노트북에서 `scene_editor.py` 파일 만들기

아래 셀을 실행하면 현재 Colab 작업 폴더에 `scene_editor.py`가 생성됩니다. 생성된 파일은 Colab 왼쪽 파일 패널에서 다운로드하거나, Google Drive에 복사해서 Windows PC로 옮기면 됩니다.

In [ ]:
from pathlib import Path

SCENE_EDITOR_CODE = r'''from __future__ import annotations

import json
import math
from datetime import datetime
from io import BytesIO
from pathlib import Path

import mujoco
import numpy as np
import streamlit as st
from PIL import Image


PACKAGE_SPECS = {
    "small_box": {"type": "box", "default_size": [0.07, 0.06, 0.045]},
    "wide_box": {"type": "box", "default_size": [0.12, 0.06, 0.045]},
    "tube_package": {"type": "cylinder", "default_size": [0.045, 0.10]},
}

PACKAGE_COLORS = {
    "target": [0.95, 0.70, 0.34, 1.0],
    "blocker": [0.78, 0.45, 0.22, 1.0],
    "irrelevant": [0.55, 0.55, 0.50, 1.0],
}

PANDA_HOME_CTRL = np.array([0.0, -0.55, 0.0, -2.20, 0.0, 1.75, 0.78, 255.0])
PANDA_HOME_QPOS = {
    "joint1": 0.0,
    "joint2": -0.55,
    "joint3": 0.0,
    "joint4": -2.20,
    "joint5": 0.0,
    "joint6": 1.75,
    "joint7": 0.78,
    "finger_joint1": 0.04,
    "finger_joint2": 0.04,
}


def camera_xyaxes_from_pos_lookat(pos, lookat, up=(0, 0, 1)):
    pos = np.array(pos, dtype=float)
    lookat = np.array(lookat, dtype=float)
    up = np.array(up, dtype=float)
    forward = lookat - pos
    forward = forward / np.linalg.norm(forward)
    z_axis = -forward
    x_axis = np.cross(up, z_axis)
    if np.linalg.norm(x_axis) < 1e-6:
        x_axis = np.array([1.0, 0.0, 0.0])
    x_axis = x_axis / np.linalg.norm(x_axis)
    y_axis = np.cross(z_axis, x_axis)
    y_axis = y_axis / np.linalg.norm(y_axis)
    return " ".join(f"{v:.6f}" for v in np.r_[x_axis, y_axis])


def package_z(shape_name, size):
    geom_type = PACKAGE_SPECS[shape_name]["type"]
    if geom_type == "box":
        return size[2]
    if geom_type == "cylinder":
        return size[1]
    return size[-1]


def package_body(name, geom_type, size, pos, rgba, mass, friction):
    size_str = " ".join(map(str, size))
    pos_str = " ".join(map(str, pos))
    rgba_str = " ".join(map(str, rgba))
    return "\n".join(
        [
            f'    <body name="{name}" pos="{pos_str}">',
            "      <freejoint/>",
            f'      <geom name="{name}_geom" type="{geom_type}" size="{size_str}" rgba="{rgba_str}" mass="{mass:.3f}" friction="{friction:.3f} 0.02 0.001"/>',
            "    </body>",
        ]
    )


def choose_action_label(target_xy, blocker_xy, irrelevant_xy):
    target_blocker_dist = float(np.linalg.norm(np.array(target_xy) - np.array(blocker_xy)))
    if target_blocker_dist < 0.22:
        return "push"
    if abs(target_xy[1]) > 0.35:
        return "reorder"
    if irrelevant_xy[0] < 0.35:
        return "ignore"
    return "pick"


def generate_xml(config):
    roles = [
        ("target", "target_package"),
        ("blocker", "blocker_package"),
        ("irrelevant", "irrelevant_package"),
    ]
    bodies = []
    objects_meta = []
    for role, body_name in roles:
        shape_name = config[f"{role}_shape"]
        spec = PACKAGE_SPECS[shape_name]
        scale = float(config[f"{shape_name}_scale"])
        size = [round(v * scale, 3) for v in spec["default_size"]]
        xy = [float(config[f"{role}_x"]), float(config[f"{role}_y"])]
        pos = [round(xy[0], 3), round(xy[1], 3), round(package_z(shape_name, size), 3)]
        rgba = PACKAGE_COLORS[role]
        bodies.append(package_body(body_name, spec["type"], size, pos, rgba, config["package_mass"], config["package_friction"]))
        objects_meta.append(
            {
                "name": body_name,
                "role": role,
                "shape": shape_name,
                "position": pos,
                "size": size,
                "rgba": rgba,
            }
        )

    camera_pos = [config["camera_x"], config["camera_y"], config["camera_z"]]
    camera_lookat = [config["lookat_x"], config["lookat_y"], config["lookat_z"]]
    light_pos = [config["light_x"], config["light_y"], config["light_z"]]
    xyaxes = camera_xyaxes_from_pos_lookat(camera_pos, camera_lookat)
    bodies_xml = "\n".join(bodies)

    xml = f"""
<mujoco model="{config['scene_id']}">
  <include file="panda.xml"/>
  <option timestep="0.005" gravity="0 0 -9.81"/>
  <statistic center="0.65 0 0.35" extent="1.6"/>
  <visual>
    <headlight diffuse="{config['light_diffuse']:.3f} {config['light_diffuse']:.3f} {config['light_diffuse']:.3f}" ambient="{config['light_ambient']:.3f} {config['light_ambient']:.3f} {config['light_ambient']:.3f}" specular="0.1 0.1 0.1"/>
    <global azimuth="135" elevation="-18"/>
  </visual>
  <asset>
    <texture type="skybox" builtin="gradient" rgb1="0.45 0.52 0.60" rgb2="0.04 0.05 0.06" width="512" height="3072"/>
    <texture type="2d" name="lobby_tile" builtin="checker" rgb1="0.78 0.78 0.74" rgb2="0.68 0.68 0.64" width="128" height="128"/>
    <material name="lobby_floor" texture="lobby_tile" texrepeat="8 8" texuniform="true" reflectance="{config['floor_reflectance']:.3f}"/>
  </asset>
  <worldbody>
    <light name="lobby_light" pos="{light_pos[0]:.3f} {light_pos[1]:.3f} {light_pos[2]:.3f}" dir="0 0 -1" directional="true"/>
    <geom name="lobby_floor" type="plane" size="3.0 3.0 0.05" material="lobby_floor"/>
{bodies_xml}
    <camera name="vlm_camera" pos="{camera_pos[0]:.3f} {camera_pos[1]:.3f} {camera_pos[2]:.3f}" xyaxes="{xyaxes}"/>
  </worldbody>
</mujoco>
""".strip()

    metadata = {
        "scene_id": config["scene_id"],
        "environment": "dorm_lobby_floor",
        "camera": "vlm_camera",
        "camera_pos": camera_pos,
        "camera_lookat": camera_lookat,
        "robot_base_pos": [config["robot_x"], config["robot_y"], 0.0],
        "robot_base_yaw": config["robot_yaw"],
        "objects": objects_meta,
        "lang_instruction": "기숙사 로비의 택배를 정리하시오.",
        "action_label": choose_action_label(
            [config["target_x"], config["target_y"]],
            [config["blocker_x"], config["blocker_y"]],
            [config["irrelevant_x"], config["irrelevant_y"]],
        ),
        "package_mass": config["package_mass"],
        "package_friction": config["package_friction"],
        "created_at": datetime.now().isoformat(timespec="seconds"),
    }
    return xml, metadata


def set_panda_base_pose(model, base_pos, base_yaw):
    for body_name in ["link0", "panda_link0"]:
        body_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, body_name)
        if body_id >= 0:
            model.body_pos[body_id] = np.array(base_pos, dtype=float)
            half = float(base_yaw) / 2.0
            model.body_quat[body_id] = np.array([math.cos(half), 0.0, 0.0, math.sin(half)])
            return


def initialize_data(model, data):
    mujoco.mj_resetData(model, data)
    for joint_name, qpos_value in PANDA_HOME_QPOS.items():
        joint_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, joint_name)
        if joint_id >= 0:
            data.qpos[model.jnt_qposadr[joint_id]] = qpos_value
    if model.nu >= 8:
        data.ctrl[:] = PANDA_HOME_CTRL
    mujoco.mj_forward(model, data)


def render_model(xml_path, metadata, width=960, height=720):
    model = mujoco.MjModel.from_xml_path(str(xml_path))
    set_panda_base_pose(model, metadata["robot_base_pos"], metadata["robot_base_yaw"])
    data = mujoco.MjData(model)
    initialize_data(model, data)
    for _ in range(160):
        mujoco.mj_step(model, data)

    with mujoco.Renderer(model, width=width, height=height) as renderer:
        renderer.update_scene(data, camera="vlm_camera")
        return renderer.render(), data


def image_download_bytes(image):
    buf = BytesIO()
    Image.fromarray(image).save(buf, format="PNG")
    return buf.getvalue()


st.set_page_config(page_title="VLA MuJoCo Lobby Scene Editor", layout="wide")
st.title("VLA MuJoCo Lobby Scene Editor")
st.caption("기숙사 로비 택배 정리 씬을 GUI로 만들고 MJCF/XML과 JSON을 저장합니다.")

with st.sidebar:
    st.header("Paths")
    panda_dir = Path(st.text_input("Panda folder", value=str(Path.cwd())))
    scene_id = st.text_input("Scene ID", value="lobby_scene_custom")

    st.header("Robot")
    robot_x = st.slider("robot x", -0.40, 0.40, 0.00, 0.01)
    robot_y = st.slider("robot y", -0.50, 0.50, 0.00, 0.01)
    robot_yaw = st.slider("robot yaw", -3.14, 3.14, 0.00, 0.01)

    st.header("Packages")
    target_shape = st.selectbox("target shape", list(PACKAGE_SPECS), index=1)
    target_x = st.slider("target x", 0.35, 1.35, 0.75, 0.01)
    target_y = st.slider("target y", -0.60, 0.60, -0.12, 0.01)

    blocker_shape = st.selectbox("blocker shape", list(PACKAGE_SPECS), index=0)
    blocker_x = st.slider("blocker x", 0.35, 1.35, 0.58, 0.01)
    blocker_y = st.slider("blocker y", -0.60, 0.60, -0.03, 0.01)

    irrelevant_shape = st.selectbox("irrelevant shape", list(PACKAGE_SPECS), index=2)
    irrelevant_x = st.slider("irrelevant x", 0.35, 1.35, 1.05, 0.01)
    irrelevant_y = st.slider("irrelevant y", -0.60, 0.60, 0.30, 0.01)

    st.header("Package Physics")
    small_box_scale = st.slider("small box scale", 0.50, 2.00, 1.00, 0.05)
    wide_box_scale = st.slider("wide box scale", 0.50, 2.00, 1.00, 0.05)
    tube_package_scale = st.slider("tube scale", 0.50, 2.00, 1.00, 0.05)
    package_mass = st.slider("package mass", 0.05, 1.00, 0.20, 0.05)
    package_friction = st.slider("package friction", 0.10, 2.00, 0.80, 0.05)

    st.header("Camera")
    camera_x = st.slider("camera x", -0.50, 2.50, 1.75, 0.01)
    camera_y = st.slider("camera y", -2.50, 0.50, -1.60, 0.01)
    camera_z = st.slider("camera z", 0.30, 2.20, 1.15, 0.01)
    lookat_x = st.slider("lookat x", 0.20, 1.30, 0.70, 0.01)
    lookat_y = st.slider("lookat y", -0.60, 0.60, 0.00, 0.01)
    lookat_z = st.slider("lookat z", 0.00, 0.80, 0.12, 0.01)

    st.header("Lighting")
    light_x = st.slider("light x", -2.00, 2.00, 0.00, 0.01)
    light_y = st.slider("light y", -3.00, 1.00, -1.00, 0.01)
    light_z = st.slider("light z", 0.80, 5.00, 2.80, 0.01)
    light_diffuse = st.slider("light diffuse", 0.10, 1.20, 0.65, 0.05)
    light_ambient = st.slider("light ambient", 0.00, 1.00, 0.35, 0.05)
    floor_reflectance = st.slider("floor reflectance", 0.00, 0.50, 0.08, 0.01)

config = {
    "scene_id": scene_id,
    "robot_x": robot_x,
    "robot_y": robot_y,
    "robot_yaw": robot_yaw,
    "target_shape": target_shape,
    "target_x": target_x,
    "target_y": target_y,
    "blocker_shape": blocker_shape,
    "blocker_x": blocker_x,
    "blocker_y": blocker_y,
    "irrelevant_shape": irrelevant_shape,
    "irrelevant_x": irrelevant_x,
    "irrelevant_y": irrelevant_y,
    "small_box_scale": small_box_scale,
    "wide_box_scale": wide_box_scale,
    "tube_package_scale": tube_package_scale,
    "package_mass": package_mass,
    "package_friction": package_friction,
    "camera_x": camera_x,
    "camera_y": camera_y,
    "camera_z": camera_z,
    "lookat_x": lookat_x,
    "lookat_y": lookat_y,
    "lookat_z": lookat_z,
    "light_x": light_x,
    "light_y": light_y,
    "light_z": light_z,
    "light_diffuse": light_diffuse,
    "light_ambient": light_ambient,
    "floor_reflectance": floor_reflectance,
}

xml, metadata = generate_xml(config)
xml_path = panda_dir / f"{scene_id}.xml"
json_path = panda_dir / f"{scene_id}.json"

col_preview, col_meta = st.columns([2, 1])
with col_preview:
    if not (panda_dir / "panda.xml").exists():
        st.warning("Panda folder에 panda.xml이 없습니다. scene_editor.py를 franka_emika_panda 폴더에서 실행하거나 sidebar 경로를 수정하세요.")
    else:
        xml_path.write_text(xml, encoding="utf-8")
        try:
            image, data = render_model(xml_path, metadata)
            metadata["robot_qpos"] = data.qpos.tolist()
            metadata["robot_qvel"] = data.qvel.tolist()
            st.image(image, caption=f"{scene_id} | label={metadata['action_label']}", use_container_width=True)
            st.download_button("Download preview PNG", data=image_download_bytes(image), file_name=f"{scene_id}.png", mime="image/png")
        except Exception as exc:
            st.error(f"Render failed: {exc}")

with col_meta:
    st.subheader("Generated files")
    if st.button("Save XML and JSON"):
        xml_path.write_text(xml, encoding="utf-8")
        json_path.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
        st.success(f"Saved: {xml_path.name}, {json_path.name}")

    st.download_button("Download XML", data=xml.encode("utf-8"), file_name=f"{scene_id}.xml", mime="text/xml")
    st.download_button(
        "Download JSON",
        data=json.dumps(metadata, ensure_ascii=False, indent=2).encode("utf-8"),
        file_name=f"{scene_id}.json",
        mime="application/json",
    )

    st.subheader("Metadata")
    st.json(metadata)
'''

scene_editor_path = Path('/content/scene_editor.py')
scene_editor_path.write_text(SCENE_EDITOR_CODE, encoding='utf-8')
print(f'생성 완료: {scene_editor_path}')
print('Colab 왼쪽 파일 패널에서 scene_editor.py를 다운로드하거나 Google Drive로 복사하세요.')

### 10-2. Windows에서 실행 준비

Windows 파일 탐색기에서 작업 폴더를 만듭니다.

```text
C:\VLA_2026\mujoco_lobby
```

PowerShell을 열고 Python이 설치되어 있는지 확인합니다.

```powershell
python --version
pip --version
```

Python이 없다면 https://www.python.org/downloads/windows/ 에서 Python 3.10 이상을 설치합니다. 설치할 때 `Add python.exe to PATH`를 체크합니다.

작업 폴더에서 가상환경을 만듭니다.

```powershell
cd C:\VLA_2026\mujoco_lobby
python -m venv .venv
.\.venv\Scripts\activate
```

필요 패키지를 설치합니다.

```powershell
pip install --upgrade pip
pip install mujoco streamlit numpy pillow
```

### 10-3. MuJoCo Menagerie 받기

Git이 설치되어 있다면:

```powershell
git clone https://github.com/google-deepmind/mujoco_menagerie.git
```

Git이 없다면 https://github.com/google-deepmind/mujoco_menagerie 에서 ZIP 파일로 다운로드해 압축을 풉니다.

최종 구조 예시:

```text
C:\VLA_2026\mujoco_lobby\mujoco_menagerie\franka_emika_panda\panda.xml
```

### 10-4. `scene_editor.py` 복사 및 실행

위 10-1에서 만든 `scene_editor.py`를 다음 위치에 복사합니다.

```text
C:\VLA_2026\mujoco_lobby\mujoco_menagerie\franka_emika_panda\scene_editor.py
```

PowerShell에서 Panda 폴더로 이동합니다.

```powershell
cd C:\VLA_2026\mujoco_lobby\mujoco_menagerie\franka_emika_panda
streamlit run scene_editor.py
```

브라우저가 열리면 왼쪽 sidebar에서 다음 항목을 조정합니다.

* 로봇 위치와 방향
* target/blocker/irrelevant 택배의 위치와 shape
* 택배 크기 scale, 질량, 마찰
* 카메라 위치와 바라보는 지점
* 조명 위치, 밝기, 바닥 반사율

화면 오른쪽에서 렌더링 결과와 metadata를 바로 확인할 수 있습니다.

### 10-5. XML/JSON 저장과 viewer 검수

GUI에서 `Save XML and JSON` 버튼을 누르면 현재 폴더에 파일이 저장됩니다.

```text
lobby_scene_custom.xml
lobby_scene_custom.json
```

공식 MuJoCo viewer로 한 번 더 확인합니다.

```powershell
python -m mujoco.viewer --mjcf=lobby_scene_custom.xml
```

viewer에서는 시뮬레이션 재생, 카메라 이동, 접촉 상황 확인을 할 수 있습니다. 단, viewer에서 수정한 내용이 자동으로 XML에 저장되는 것은 아닙니다. 수정은 `scene_editor.py`에서 다시 하는 것을 기본으로 합니다.

### 10-6. Colab으로 가져오기

수정이 끝난 XML/JSON을 Google Drive 팀 폴더나 GitHub에 업로드합니다. Google Drive를 쓰는 경우 Colab에서 다음처럼 복사합니다.

```python
import shutil

src_xml = '/content/drive/MyDrive/VLA_Team_2026/data/lobby_scene_custom.xml'
src_json = '/content/drive/MyDrive/VLA_Team_2026/data/lobby_scene_custom.json'

dst_xml = str(PANDA_DIR / 'lobby_scene_custom.xml')
dst_json = str(JSON_DIR / 'lobby_scene_custom.json')

shutil.copy2(src_xml, dst_xml)
shutil.copy2(src_json, dst_json)
```

불러온 XML을 다시 렌더링합니다.

```python
with open(dst_json, encoding='utf-8') as f:
    metadata = json.load(f)

model, data = load_lobby_model(dst_xml, metadata)
for _ in range(160):
    mujoco.mj_step(model, data)

image = render_rgb(model, data, camera='vlm_camera')
show_rgb(image, 'Windows editor에서 만든 씬')
```

### 10-7. 흔한 오류

* `panda.xml not found`: `scene_editor.py`와 XML이 `panda.xml`이 있는 폴더에서 실행되는지 확인합니다.
* 브라우저가 열리지 않음: PowerShell 출력에 있는 `Local URL`을 직접 브라우저에 붙여넣습니다.
* 렌더링 실패: `pip install mujoco streamlit numpy pillow`가 같은 가상환경에 설치되어 있는지 확인합니다.
* 장면이 검게 보임: 카메라가 바닥이나 빈 공간을 보고 있을 수 있습니다. `camera`와 `lookat` 값을 조정합니다.
* 물체가 튐: 물체끼리 겹쳐 시작했거나, 로봇과 택배가 겹쳤을 수 있습니다. 위치를 조금 벌리고 다시 저장합니다.

이 workflow의 목표는 XML 숫자를 직접 외워서 고치는 것이 아니라, **GUI로 장면을 설계하고, XML/JSON이 어떤 실험 데이터로 저장되는지 이해하는 것**입니다.

자습용 참고 오픈소스
* https://github.com/yunhaif/reflect-vlm
* https://github.com/jeongeun980906/lerobot-mujoco-tutorial